In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv(override=True)

from pathwiseagenticassistant.tools.web_search_agent import web_search_agent
from pathwiseagenticassistant.tools.summarizer import summarizer_agent
from pathwiseagenticassistant.agents.research.research_manager import ResearchManager
from pathwiseagenticassistant.schemas import LearningRequest, LevelEnum

print("imports OK")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from agents import Runner, trace
from pathwiseagenticassistant.agents.research.topic_planning_agent import topic_planning_agent

request_text = """
topic: Apache Kafka
level: intermediate
tech_stack: Java, Spring Boot
goal: build event-driven microservices at work
"""

with trace("Pathwise - topic planner test"):
    result = await Runner.run(topic_planning_agent, request_text)

plan = result.final_output
print("Subtopics:", plan.subtopics)
for s in plan.searches:
    print(f"[{s.angle}] {s.query}")
    print(f"     why: {s.reason}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Subtopics: ['Kafka Architecture', 'Producers and Consumers', 'Kafka Topics and Partitions', 'Kafka Streams API', 'Integrating Kafka with Spring Boot']
[what_it_is] What is Apache Kafka and how does it work
     why: Understanding the fundamentals of Kafka for event-driven design.
[problem_it_solves] What problem does Apache Kafka solve
     why: Identify the challenges Kafka is designed to address in distributed systems.
[pros_and_cons] Apache Kafka pros and cons
     why: Evaluate Kafka's benefits and limitations for microservices.
[real_world_example] Real examples of Apache Kafka usage
     why: Learn from case studies of companies successfully implementing Kafka.
[how_to] How to use Apache Kafka with Java Spring Boot
     why: Get practical guidance on integrating Kafka into your tech stack.


In [3]:
from pathwiseagenticassistant.agents.research.relevance_judge import relevance_judge_agent, MIN_RELEVANCE_SCORE

learner = "topic: Apache Kafka, level: intermediate, stack: Java Spring Boot, goal: microservices"

fake_articles = [
    "title: Spring for Apache Kafka Reference | url: https://docs.spring.io/spring-kafka/reference/ | snippet: Official guide to producers, consumers and listeners in Spring Boot.",
    "title: Learn Kafka in 7 Days - 80% OFF Today! | url: https://best-course-deals.example.com/kafka | snippet: Buy our bestselling course now, limited offer.",
    "title: The History of Franz Kafka's Novels | url: https://literature.example.com/kafka | snippet: An overview of the Czech writer's major works.",
]

with trace("Pathwise - judge test"):
    for article in fake_articles:
        result = await Runner.run(relevance_judge_agent, f"LEARNER: {learner}\nARTICLE: {article}")
        v = result.final_output
        decision = "KEEP" if v.score >= MIN_RELEVANCE_SCORE else "DROP"
        print(f"{decision} | score {v.score} | {v.reason}")

KEEP | score 5 | The article is an official guide that directly addresses using Apache Kafka with Spring Boot, perfectly aligning with the learner's intermediate level and goal of microservices.
DROP | score 1 | The article is an advertisement for a course and does not provide any relevant information on Apache Kafka.
DROP | score 1 | The article is unrelated to Apache Kafka and focuses on a literary figure instead.


In [4]:
from pathwiseagenticassistant.agents.research.critic_agent import critic_agent

critic_input = """
LEARNER: topic: Apache Kafka, level: intermediate, stack: Java Spring Boot, goal: microservices

PLANNED SUBTOPICS:
- Kafka architecture
- Topics and partitions
- Producers and consumers
- Consumer groups
- Spring Boot integration

ARTICLE NOTES:
1. Title: Kafka Introduction (kafka.apache.org)
   Key points: Kafka is a distributed event log; brokers store topics; topics are split into partitions.
   Subtopics covered: Kafka architecture, topics and partitions

2. Title: Spring for Apache Kafka Reference (docs.spring.io)
   Key points: KafkaTemplate sends messages; @KafkaListener receives them; configure with application.yml.
   Subtopics covered: producers and consumers, Spring Boot integration
"""

with trace("Pathwise - critic test"):
    result = await Runner.run(critic_agent, critic_input)

report = result.final_output
print("Missing:", report.missing_subtopics)
print("Contradictions:", report.contradictions)
print("Follow-up searches:", report.follow_up_queries)

Missing: ['Consumer groups']
Contradictions: []
Follow-up searches: ['Consumer groups in Apache Kafka']


In [8]:
from pydantic import BaseModel
from agents import Agent, function_tool
from ddgs import DDGS

# ----- TEMP web search (until John's is ready) -----
class Article(BaseModel):
    title: str
    url: str
    snippet: str

class SearchResponse(BaseModel):
    query: str
    articles: list[Article]

@function_tool
def duckduckgo_search(query: str) -> str:
    """Search the web and return title, URL and snippet for the top results."""
    try:
        results = DDGS().text(query, max_results=5)
        return "\n\n".join(
            f"Title: {r['title']}\nURL: {r['href']}\nSnippet: {r['body']}" for r in results
        ) or "No results."
    except Exception as e:
        return f"Search failed: {e}"

temp_search_agent = Agent(
    name="TEMP Web Search",
    instructions="Call duckduckgo_search with the query. Return up to 4 real results. Never invent URLs.",
    tools=[duckduckgo_search],
    output_type=SearchResponse,
    model="gpt-4o-mini",
)

# ----- TEMP summarizer (until John's is ready) -----
class TempSummary(BaseModel):
    title: str
    url: str
    key_points: list[str]
    subtopics_covered: list[str]

temp_summarizer_agent = Agent(
    name="TEMP Summarizer",
    instructions="From the article's title and snippet, write 3-5 key points and list the subtopics it covers.",
    output_type=TempSummary,
    model="gpt-4o-mini",
)

In [9]:
from pathwiseagenticassistant.agents.research.research_manager import ResearchManager, LearningRequest

request = LearningRequest(
    topic="Apache Kafka",
    user_level="intermediate",
    tech_stack="Java, Spring Boot",
    goal="build event-driven microservices at work",
)

manager = ResearchManager(temp_search_agent, temp_summarizer_agent)
pack = await manager.run(request)

print("Topic:", pack.topic)
print("Gaps:", pack.gaps)
for s in pack.summaries:
    print()
    print("-", s.title)
    print(" ", s.url)
    for point in s.key_points:
        print("    •", point)

Topic: Apache Kafka
Gaps: ['Producers and Consumers', 'Kafka Streams', 'Schema Registry', 'Integrating Kafka with Spring Boot', 'Monitoring and Managing Kafka', 'Event Sourcing with Kafka']

- Apache Kafka Overview
  https://en.wikipedia.org/wiki/Apache_Kafka
    • Apache Kafka is a distributed messaging system that ensures message ordering within partitions.
    • It operates on a log-based architecture, retaining messages in a durable, append-only log.
    • Multiple consumers can read messages independently from different offsets, unlike traditional queue-based systems.
    • Kafka is designed for high throughput and scalability, making it suitable for large-scale systems.
    • It supports stream processing, allowing real-time data processing and analytics.

- TempSummary
  https://softwaremill.com/what-problems-does-kafka-solve-in-distributed-systems/
    • Kafka provides a robust architecture for managing distributed data communication.
    • It enhances scalability and resilienc